# Task 2: Predicting Rainfall
## Singapore Rainfall, 2017–2024

## 2.1 Spark Session

In [1]:
import socket

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

from pyspark.sql import SparkSession
from pyspark.sql import Window

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pyspark.sql.functions as F
import seaborn as sns
import pandas as pd

import os
import sys

In [2]:
spark = SparkSession.builder.getOrCreate()

spark.conf.set("spark.sql.session.timeZone", "Asia/Singapore")

print("Running on:   ", socket.gethostname())
print("Spark master: ", spark.sparkContext.master)
print("Spark version:", spark.version)

Running on:    rainfall-dev-m
Spark master:  yarn
Spark version: 3.5.3


In [3]:
# Constants for Central Coordinates and Rainfall Multiplier per Day
# There are 12 five-minute intervals per hour and 24 hours per day
C_LAT, C_LON = 1.3521, 103.8198 
RF_MULTIPLIER = (24 * 60) / 5

## 2.2 Reading the Cleaned Rainfall Data

In [4]:
rain = spark.read.parquet("gs://dsa5208-rainfall-sixing-data/parquet/")

In [5]:
# Step 1: Extract the year and month from the observation date
rain = (
    rain.withColumn("date", F.to_date("date"))
        .withColumn("year", F.year("date"))
        .withColumn("month", F.month("date"))
)

# Step 2: Calculate mean rainfall per observed five-minute interval
# for each year and month
monthly_spk = (
    rain.filter(F.col("year").between(2017, 2024))
        .groupBy("year", "month")
        .agg(
            F.avg("reading_value").alias("mean_per_interval"),
            F.countDistinct("station_id").alias("n_stations")
        )
        .withColumn(
            "year_month",
            F.make_date("year", "month", F.lit(1))
        )
        .withColumn(
            "days",
            F.dayofmonth(F.last_day("year_month"))
        )
        # mean interval rainfall × expected intervals in the month
        .withColumn(
            "avg_rainfall_monthly",
            F.col("mean_per_interval") * F.col("days") * RF_MULTIPLIER
        )
        .orderBy("year_month")
)

## 2.3 Auto-Correlation of Rainfall and Its Timelags 

In [6]:
def lookback(seconds):
    return (Window.partitionBy("station_id")
            .orderBy("ts_long")
            .rangeBetween(-seconds, 0))

In [7]:
rain_lag = rain.withColumn("ts_long", F.col("timestamp").cast("long"))

w = Window.partitionBy("station_id").orderBy("ts_long")

lags = {"lag_5m": 1, "lag_30m": 6, "lag_1h": 12}

for name, k in lags.items():
    rain_lag = (rain_lag
        .withColumn(name, F.lag("reading_value", k).over(w))
        .withColumn(name + "_ts", F.lag("ts_long", k).over(w)))
    
    # null out where the gap isn't the expected duration
    rain_lag = rain_lag.withColumn(name,
        F.when(F.col("ts_long") - F.col(name + "_ts") == k * 300,
               F.col(name)))
    rain_lag = rain_lag.drop(name + "_ts")

In [8]:
## The target label
future_30m = (Window.partitionBy("station_id")
                .orderBy("ts_long")
                .rangeBetween(1, 1800))

rain_lag = (rain_lag
    # rolling sums
    .withColumn("rain_cumsum_30m", F.sum("reading_value").over(lookback(1800)))
    .withColumn("rain_cumsum_1h",  F.sum("reading_value").over(lookback(3600)))
    # time features
    .withColumn("hour", F.hour("timestamp"))
    # target
    .withColumn("future_30m", F.sum("reading_value").over(future_30m)))

In [14]:
feature_cols = [
    "location_latitude", "location_longitude",
    "hour", "month",
    "reading_value",
    "lag_5m", "lag_30m", "lag_1h",
    "rain_cumsum_30m", "rain_cumsum_1h",
]

THRESHOLD = 0.5

feature_df = (rain_lag.filter(F.col("future_30m").isNotNull())
              .filter(F.col("lag_1h").isNotNull())
              .withColumn("label", (F.col("future_30m") > THRESHOLD).cast("int")))

train = feature_df.filter(F.col("year") <= 2022)
val   = feature_df.filter(F.col("year") == 2023)
test  = feature_df.filter(F.col("year") == 2024)

train.select(F.avg("label").alias("positive_rate")).show()

+--------------------+
|       positive_rate|
+--------------------+
|0.033587661294852786|
+--------------------+



In [15]:
assembler = VectorAssembler(inputCols=feature_cols,
                            outputCol="features",
                            handleInvalid="skip")

gbt = GBTClassifier(featuresCol="features", labelCol="label",
                    maxDepth=5, maxIter=50, stepSize=0.1, seed=42)

train_v = assembler.transform(train)
test_v  = assembler.transform(test)

model = gbt.fit(train_v)
preds = model.transform(test_v)

pr  = BinaryClassificationEvaluator(metricName="areaUnderPR",
        labelCol="label", rawPredictionCol="rawPrediction").evaluate(preds)
roc = BinaryClassificationEvaluator(metricName="areaUnderROC",
        labelCol="label", rawPredictionCol="rawPrediction").evaluate(preds)

print(f"PR-AUC  {pr:.4f}   (baseline {0.023:.4f})")
print(f"ROC-AUC {roc:.4f}")

PR-AUC  0.6387   (baseline 0.0230)
ROC-AUC 0.8935


In [ ]:
# feature importances
for name, imp in sorted(zip(feature_cols, model.featureImportances.toArray()),
                        key=lambda x: -x[1]):
    print(f"{imp:.4f}  {name}")

# persistence baseline 
naive = preds.withColumn("naive", (F.col("rain_cumsum_30m") > 0.5).cast("int"))
naive.groupBy("naive", "label").count().orderBy("naive", "label").show()

0.4837  rain_cumsum_30m
0.3491  reading_value
0.0506  hour
0.0499  rain_cumsum_1h
0.0244  month
0.0229  lag_5m
0.0130  lag_30m
0.0062  lag_1h
0.0001  location_longitude
0.0000  location_latitude


Row(avg(CAST((naive = label) AS INT))=0.9670901466257842)


In [18]:
naive = preds.withColumn("naive", (F.col("rain_cumsum_30m") > 0.5).cast("int"))

naive.groupBy("naive", "label").count().orderBy("naive", "label").show()

+-----+-----+-------+
|naive|label|  count|
+-----+-----+-------+
|    0|    0|6006697|
|    0|    1|  93060|
|    1|    0| 116011|
|    1|    1| 137071|
+-----+-----+-------+



In [19]:
preds.groupBy("prediction", "label").count().orderBy("prediction", "label").show()

+----------+-----+-------+
|prediction|label|  count|
+----------+-----+-------+
|       0.0|    0|6097169|
|       0.0|    1| 121179|
|       1.0|    0|  25539|
|       1.0|    1| 108952|
+----------+-----+-------+



In [21]:
base_rate = train.select(F.avg("label")).first()[0]

shuffled = train.withColumn("label",
    F.col("label") * 0 + (F.rand(seed=7) < base_rate).cast("int"))

m2 = gbt.fit(assembler.transform(shuffled))
p2 = m2.transform(assembler.transform(test))
print(BinaryClassificationEvaluator(metricName="areaUnderPR",
        labelCol="label", rawPredictionCol="rawPrediction").evaluate(p2))

0.1974824343394064


In [22]:
print(BinaryClassificationEvaluator(metricName="areaUnderROC",
        labelCol="label", rawPredictionCol="rawPrediction").evaluate(p2))

0.5637655553184941


26/10/11 16:21:28 ERROR TransportResponseHandler: Still have 1 requests outstanding when connection from /10.148.0.33:45252 is closed
26/10/11 16:21:28 ERROR TransportClient: Failed to send RPC RPC 8693474043611835936 to /10.148.0.33:45252: io.netty.channel.StacklessClosedChannelException
io.netty.channel.StacklessClosedChannelException: null
	at io.netty.channel.AbstractChannel$AbstractUnsafe.write(Object, ChannelPromise)(Unknown Source) ~[netty-transport-4.1.100.Final.jar:4.1.100.Final]
26/10/11 16:21:28 WARN BlockManagerMasterEndpoint: Error trying to remove RDD 2211 from block manager BlockManagerId(31, rainfall-dev-w-0.asia-southeast1-b.c.dsa5208-rainfall-sixing.internal, 40695, None)
java.io.IOException: Connection from /10.148.0.33:45252 closed
	at org.apache.spark.network.client.TransportResponseHandler.channelInactive(TransportResponseHandler.java:147) ~[spark-network-common_2.12-3.5.3.jar:3.5.3]
	at org.apache.spark.network.server.TransportChannelHandler.channelInactive(Trans